---
title: Llama 3.3 70b steering benchmarks
jupyter:
  jupytext:
    text_representation:
      extension: .qmd
      format_name: quarto
      format_version: '1.0'
      jupytext_version: 1.19.3
  kernelspec:
    display_name: Python 3
    name: python3
    language: python
---


This notebook clones the repo from GitHub, loads `meta-llama/Llama-3.3-70B-Instruct`, attaches the steering hook, and runs one `lm-evaluation-harness` cell per benchmark.

Defaults use `ZeroSteerer` so you can compare against a no-op steered wrapper before swapping in a real steering method.

In the future, we would like to implement a live steerer using CAM-compatible algorithms, which is simple to test using this notebook.


In [1]:
#@title Clone Repo
REPO_URL = "https://github.com/JDub323/cam_enabled_steering_llm.git"
REPO_DIR = "cam_enabled_steering_llm"

import os
if not os.path.exists(REPO_DIR):
    !git clone -q $REPO_URL
%cd $REPO_DIR
!pip install -q -e ".[eval]"

/content/cam_enabled_steering_llm
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 6.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 154.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 12.1 MB/s eta 0:00:00
  Building editable for gemma2-steering-research (pyproject.toml) ... done


In [2]:
#@title Authenticate for gated Llama weights
from huggingface_hub import login
login()

In [ ]:
#@title Download Llama 3.3 70B, get Steered Model
# this takes a while to download (70B params) and load across GPUs.
import torch
import lm_eval
from lm_eval.tasks import TaskManager

from gemma2_steering import SteeredGemma2, ZeroSteerer, ConstantSteerer, as_lm_eval_model

MODEL_NAME = "meta-llama/Llama-3.3-70B-Instruct"
LAYER = 12
HOOK_POINT = "layer_output"  # options: layer_output, post_attention, post_mlp

model = SteeredGemma2(
    ZeroSteerer(),
    model_name=MODEL_NAME,
    layer=LAYER,
    hook_point=HOOK_POINT,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

In [ ]:
#@title Benchmark Configs
SAMPLE_LIMIT=50 # set to None to run full benchmarks
NUM_FEWSHOT=0

In [ ]:
# @title Make a model harness
# Running on an H100; batch_size='auto' will pick an appropriate batch size for the 70B model.
lm = as_lm_eval_model(model, tokenizer=model.tokenizer, batch_size='auto')
task_manager = TaskManager(include_path="benchmarks")

In [ ]:
#@title Helper Functions
import gc
import torch

def run_eval(task, *, limit=20, num_fewshot=0):
    """Run one lm-eval task. Keep limit small on Colab; set limit=None for full evals."""
    # Clear memory before running the evaluation to prevent fragmented OOMs
    gc.collect()
    torch.cuda.empty_cache()

    result = lm_eval.simple_evaluate(
        model=lm,
        tasks=[task],
        task_manager=task_manager,
        limit=limit,
        num_fewshot=num_fewshot,
        log_samples=False,
    )

    # Clear memory after the evaluation
    gc.collect()
    torch.cuda.empty_cache()

    return result


def try_first_available(tasks, *, limit=20, num_fewshot=0):
    errors = {}
    for task in tasks:
        try:
            print(f"Trying {task!r}...")
            return run_eval(task, limit=limit, num_fewshot=num_fewshot)
        except Exception as exc:
            errors[task] = repr(exc)
    print("None of the candidate task names worked. Inspect installed tasks with:")
    print("!lm-eval ls tasks | grep -Ei 'hle|humanity|last_exam'")
    raise RuntimeError(errors)

In [ ]:
#@title Generation sanity check.
prompt = "The capital of France is"
inputs = model.tokenizer(prompt, return_tensors="pt").to(model.device)
output = model.generate(**inputs, max_new_tokens=16)
print(model.tokenizer.decode(output[0], skip_special_tokens=True))

# Disable steering without reloading Llama:
with model.without_steering():
    output = model.generate(**inputs, max_new_tokens=16)
print(model.tokenizer.decode(output[0], skip_special_tokens=True))

## Benchmark: MMLU

In [ ]:
# MMLU is large; use a small limit for Colab smoke tests.
mmlu_results = run_eval("mmlu", limit=SAMPLE_LIMIT, num_fewshot=NUM_FEWSHOT)
mmlu_results["results"]

## Benchmark: Humanity's Last Exam (DOES NOT EXIST YET)

In [ ]:
# HLE availability depends on the installed lm-eval release.
# This tries common task names and prints an inspection command if unavailable.
# hle_results = try_first_available(
#     ["humanitys_last_exam", "humanity_last_exam", "hle", "hle_text"],
#     limit=10,
#     num_fewshot=0,
# )
# hle_results["results"]

## Benchmark: TruthfulQA

In [ ]:
truthfulqa_results = run_eval("truthfulqa_mc2", limit=SAMPLE_LIMIT, num_fewshot=NUM_FEWSHOT)
truthfulqa_results["results"]

## Benchmark: Sycophancy (IS BROKEN)

In [ ]:
# Local YAML in benchmarks/sycophancy_on_nlp_survey.yaml.
# sycophancy_results = run_eval("sycophancy_on_nlp_survey", limit=50, num_fewshot=0)
# sycophancy_results["results"]

## Benchmark: WMDP dangerous-knowledge / malicious-capability probe

In [ ]:
# WMDP covers biosecurity, cybersecurity, and chemical-security multiple-choice questions.
wmdp_results = run_eval("wmdp", limit=SAMPLE_LIMIT, num_fewshot=NUM_FEWSHOT)
wmdp_results["results"]

## Swap in a constant vector

`ConstantSteerer` requires a batched vector. This is intentionally explicit; use `[1, hidden_size]` for a vector shared across the batch.

In [ ]:
hidden = model.config.hidden_size
constant = torch.zeros(1, hidden, device=model.device, dtype=torch.bfloat16)
constant[:, 0] = 1.0

model.steerer = ConstantSteerer(constant)
# Re-run any benchmark cell above to compare against ZeroSteerer.

## View Results
Print the results using `lm_eval`'s built-in table formatter.

In [ ]:
from lm_eval.utils import make_table

print("=== MMLU Results ===")
if 'mmlu_results' in globals():
    print(make_table(mmlu_results))

print("\n=== TruthfulQA Results ===")
if 'truthfulqa_results' in globals():
    print(make_table(truthfulqa_results))

print("\n=== WMDP Results ===")
if 'wmdp_results' in globals():
    print(make_table(wmdp_results))

In [ ]:
def print_summary_accuracy(results_dict, suite_name):
    if not results_dict or 'results' not in results_dict:
        return

    results = results_dict['results']
    acc_values = []

    # Try to find the exact suite name first (aggregate score)
    for task_name, metrics in results.items():
        # lm_eval uses various keys for accuracy depending on the task
        acc = metrics.get('acc,none') or metrics.get('acc_norm,none') or metrics.get('mc2') or metrics.get('acc')

        if acc is not None:
            # If it's an aggregate key (e.g., 'mmlu' or 'wmdp'), print it directly
            if task_name == suite_name.lower():
                print(f"{suite_name} Overall Accuracy: {acc * 100:.2f}%")
                return
            acc_values.append(acc)

    # If no single aggregate key is found, average the subtasks
    if acc_values:
        overall_acc = sum(acc_values) / len(acc_values)
        print(f"{suite_name} Overall Accuracy (averaged): {overall_acc * 100:.2f}%")
    else:
        print(f"{suite_name}: No standard accuracy metric found.")

print("=== Benchmark Accuracy Summaries ===")
if 'mmlu_results' in globals():
    print_summary_accuracy(mmlu_results, "MMLU")
if 'truthfulqa_results' in globals():
    # TruthfulQA usually reports 'mc2' or 'acc'
    print_summary_accuracy(truthfulqa_results, "TruthfulQA")
if 'wmdp_results' in globals():
    print_summary_accuracy(wmdp_results, "WMDP")